
# Notebook 45 — Governing Equation Learning for Residual Flow

This notebook follows Notebook 44 by moving from **phase description** to **governing-equation learning**.

We model the residual flow as a non-autonomous 1D dynamical system:

\[
\frac{dr}{dc} = g(r,c)
\]

Main goals:

1. Load a real export from Notebook 43/44 when present.
2. Otherwise generate a synthetic fallback dataset with the same schema.
3. Fit several surrogate models for \(g(r,c)\):
   - linear baseline
   - polynomial regression
   - RBF / kernel-style local regression
   - small MLP regressor
4. Test **separability**:
   - additive: \(g(r,c) \approx a(r) + b(c)\)
   - multiplicative low-rank approximation
5. Compare:
   - pointwise prediction quality
   - integration quality
   - forward/backward asymmetry
6. Produce a compact summary table and a recommended governing surrogate.

The ideal outcome is not merely high predictive \(R^2\), but a model that preserves the observed transport structure under integration.


In [ ]:

import os, glob, warnings, math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.kernel_ridge import KernelRidge

warnings.filterwarnings("ignore")
np.random.seed(42)


In [ ]:

# ----------------------------
# Configuration
# ----------------------------

DATA_PATH = None  # set manually if desired
USE_SYNTHETIC_FALLBACK = True

FOCUS = {
    "system": "entropy",
    "task": "zeta_vs_gue",
    "forcing_mode": "condition_gap",
    "k": 5,
    "flow_mode": "nonlinear",
}

TEST_SIZE = 0.25
RANDOM_STATE = 42

# Integration controls
INTEGRATION_GRID_SIZE = 60
R0_COUNT = 35

print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)
print("FOCUS:", FOCUS)


In [ ]:

# ----------------------------
# Synthetic fallback generator
# ----------------------------

def synthetic_flow_core(r, c, system="entropy", task="zeta_vs_gue",
                        forcing_mode="condition_gap", k=5, flow_mode="nonlinear"):
    # Base restoring + transport structure
    restoring = -0.85 * (r - 0.18 * np.tanh(1.8 * c))
    transport = 0.35 * np.sin(1.7 * c) + 0.12 * c
    curvature = -0.22 * r**3 + 0.18 * r * np.cos(1.2 * c)

    if system == "unevenness":
        restoring *= 0.9
        transport += 0.05 * np.sin(3.0 * c)
        curvature += 0.05 * r**2 * np.sign(c)

    if task == "zeta_vs_poisson":
        transport += 0.08 * np.cos(2.3 * c)
        curvature += 0.04 * c**2 * np.sign(r)

    if forcing_mode == "capacity_gap":
        forcing = 0.08 * np.sin(2.5 * c) - 0.03 * r
    elif forcing_mode == "feature_gap":
        forcing = 0.06 * np.cos(2.0 * c) + 0.05 * r * c
    else:  # condition_gap
        forcing = 0.14 * np.tanh(1.4 * c) + 0.09 * r * np.sin(c)

    scale = {3: 0.92, 5: 1.0, 7: 0.88}.get(int(k), 1.0)

    if flow_mode == "linear":
        g = scale * (-0.95 * r + 0.22 * c + forcing)
    else:
        g = scale * (restoring + transport + curvature + forcing)

    return g


def generate_synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0

    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        c_grid = np.linspace(-1.25, 1.05, 28)
                        r0_values = np.linspace(-0.95, 0.60, 21)

                        for path_id, r0 in enumerate(r0_values):
                            r = float(r0)
                            prev_r = r
                            for window_id, c in enumerate(c_grid):
                                dc = 0.04 + 0.06 * np.random.rand()
                                g = synthetic_flow_core(
                                    r, c, system=system, task=task,
                                    forcing_mode=forcing_mode, k=k, flow_mode=flow_mode
                                )
                                noise = 0.02 * np.random.randn()
                                g_noisy = g + noise
                                next_r = r + dc * g_noisy
                                jacobian_norm = abs(-0.85 - 0.66 * r**2 + 0.18 * np.cos(1.2 * c))
                                integration_error = abs(next_r - (r + dc * g))
                                rows.append({
                                    "system": system,
                                    "task": task,
                                    "forcing_mode": forcing_mode,
                                    "k": k,
                                    "flow_mode": flow_mode,
                                    "condition_coord": c,
                                    "residual": r,
                                    "predicted_flow": g_noisy,
                                    "next_residual": next_r,
                                    "delta_condition": dc,
                                    "sample_id": sample_id,
                                    "path_id": path_id,
                                    "window_id": window_id,
                                    "jacobian_norm": jacobian_norm,
                                    "integration_error": integration_error,
                                    "prev_residual": prev_r,
                                    "true_flow_synth": g,
                                })
                                prev_r = r
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)


In [ ]:

# ----------------------------
# Load real data or fallback
# ----------------------------

def auto_find_data():
    candidates = []
    patterns = [
        "*.parquet", "*.csv", "*.pkl", "*.pickle",
        "data/*.parquet", "data/*.csv", "outputs/*.parquet", "outputs/*.csv"
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
    return sorted(set(candidates))

def read_any(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    if path.endswith(".csv"):
        return pd.read_csv(path)
    if path.endswith(".pkl") or path.endswith(".pickle"):
        return pd.read_pickle(path)
    raise ValueError(f"Unsupported file type: {path}")

def standardize_columns(df):
    alias_groups = {
        "condition_coord": ["condition_coord", "c", "condition", "condition_coordinate", "cond", "mean_c"],
        "residual": ["residual", "r", "resid", "mean_r"],
        "predicted_flow": ["predicted_flow", "flow", "g", "mean_flow", "drdc", "dr_dc"],
        "next_residual": ["next_residual", "r_next", "next_r"],
        "delta_condition": ["delta_condition", "dc", "delta_c", "step_c"],
        "system": ["system"],
        "task": ["task"],
        "forcing_mode": ["forcing_mode", "forcing"],
        "k": ["k", "window", "window_size"],
        "flow_mode": ["flow_mode", "model_mode", "mode"],
        "path_id": ["path_id", "trajectory_id"],
        "window_id": ["window_id", "step_id"],
        "sample_id": ["sample_id", "id"],
    }
    df = df.copy()
    for target, aliases in alias_groups.items():
        for a in aliases:
            if a in df.columns:
                if a != target:
                    df[target] = df[a]
                break

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after standardization: {missing}")

    # Fill defaults for optional columns.
    if "system" not in df.columns: df["system"] = "unknown_system"
    if "task" not in df.columns: df["task"] = "unknown_task"
    if "forcing_mode" not in df.columns: df["forcing_mode"] = "unknown_forcing"
    if "k" not in df.columns: df["k"] = -1
    if "flow_mode" not in df.columns: df["flow_mode"] = "unknown_flow_mode"
    if "delta_condition" not in df.columns: df["delta_condition"] = 1.0
    if "path_id" not in df.columns: df["path_id"] = 0
    if "window_id" not in df.columns: df["window_id"] = np.arange(len(df))
    if "sample_id" not in df.columns: df["sample_id"] = np.arange(len(df))
    if "next_residual" not in df.columns:
        df["next_residual"] = df["residual"] + df["delta_condition"] * df["predicted_flow"]

    return df

data_source = None
if DATA_PATH is None:
    found = auto_find_data()
    if found:
        DATA_PATH = found[0]

if DATA_PATH is not None:
    print("Loading:", DATA_PATH)
    df = read_any(DATA_PATH)
    df = standardize_columns(df)
    data_source = DATA_PATH
else:
    if not USE_SYNTHETIC_FALLBACK:
        raise FileNotFoundError("No data file found and synthetic fallback disabled.")
    print("No candidate export found. Generating synthetic fallback dataset...")
    df = generate_synthetic_dataset()
    df = standardize_columns(df)
    data_source = "synthetic_fallback"

print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())


In [ ]:

# Focus subset
focus_df = df.copy()
for key, value in FOCUS.items():
    if key in focus_df.columns:
        focus_df = focus_df[focus_df[key] == value]

if len(focus_df) < 50:
    print("Warning: focused subset small. Falling back to first compatible slice.")
    group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
    counts = df.groupby(group_cols).size().reset_index(name="n").sort_values("n", ascending=False)
    fallback = counts.iloc[0].to_dict()
    print("Fallback slice:", fallback)
    focus_df = df.copy()
    for key in group_cols:
        focus_df = focus_df[focus_df[key] == fallback[key]]

focus_df = focus_df.sort_values(["path_id", "window_id", "condition_coord"]).reset_index(drop=True)
print("Focus shape:", focus_df.shape)
display(focus_df.head())


In [ ]:

# ----------------------------
# Train / test split
# ----------------------------

feature_cols = ["residual", "condition_coord"]
target_col = "predicted_flow"

X = focus_df[feature_cols].values
y = focus_df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print("Train size:", len(X_train), "Test size:", len(X_test))


In [ ]:

# ----------------------------
# Model definitions
# ----------------------------

models = {
    "linear": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", LinearRegression())
    ]),
    "poly_deg2": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0))
    ]),
    "poly_deg3": Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("scaler", StandardScaler()),
        ("reg", Ridge(alpha=1.0))
    ]),
    "kernel_rbf": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", KernelRidge(alpha=0.2, kernel="rbf", gamma=0.9))
    ]),
    "mlp_small": Pipeline([
        ("scaler", StandardScaler()),
        ("reg", MLPRegressor(hidden_layer_sizes=(32, 32), activation="tanh",
                             max_iter=2500, random_state=RANDOM_STATE))
    ]),
}


In [ ]:

# ----------------------------
# Fit and evaluate pointwise prediction
# ----------------------------

def regression_metrics(y_true, y_pred):
    return {
        "r2": r2_score(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
    }

model_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    model_results[name] = {
        "model": model,
        "train_metrics": regression_metrics(y_train, pred_train),
        "test_metrics": regression_metrics(y_test, pred_test),
    }

results_rows = []
for name, d in model_results.items():
    row = {"model_name": name}
    for split in ["train_metrics", "test_metrics"]:
        for k, v in d[split].items():
            row[f"{split}_{k}"] = v
    results_rows.append(row)

results_df = pd.DataFrame(results_rows).sort_values("test_metrics_r2", ascending=False)
display(results_df)


In [ ]:

# Plot predicted vs actual for top models
top_models = list(results_df["model_name"].head(4))

fig = plt.figure(figsize=(8, 6))
for i, name in enumerate(top_models, 1):
    model = model_results[name]["model"]
    yhat = model.predict(X_test)
    plt.scatter(y_test, yhat, s=12, alpha=0.45, label=name)

mn = min(y_test.min(), *(model_results[n]["model"].predict(X_test).min() for n in top_models))
mx = max(y_test.max(), *(model_results[n]["model"].predict(X_test).max() for n in top_models))
plt.plot([mn, mx], [mn, mx], linestyle="--")
plt.xlabel("actual flow")
plt.ylabel("predicted flow")
plt.title("Pointwise governing-flow prediction")
plt.legend()
plt.show()



## Separability tests

Two useful structural probes:

1. **Additive separability**
   \[
   g(r,c) \approx a(r) + b(c)
   \]

2. **Low-rank multiplicative structure**
   by reshaping a gridded estimate of \(g\) and examining singular values.

These do not prove the true structure, but they indicate whether a compact symbolic form may exist.


In [ ]:

# ----------------------------
# Additive separability test
# ----------------------------

def fit_additive_model(df_slice, n_r_bins=24, n_c_bins=24):
    tmp = df_slice[["residual", "condition_coord", "predicted_flow"]].copy()
    tmp["r_bin"] = pd.cut(tmp["residual"], bins=n_r_bins, labels=False, duplicates="drop")
    tmp["c_bin"] = pd.cut(tmp["condition_coord"], bins=n_c_bins, labels=False, duplicates="drop")
    grid = tmp.groupby(["r_bin", "c_bin"], observed=False)["predicted_flow"].mean().reset_index()

    nr = int(grid["r_bin"].max() + 1)
    nc = int(grid["c_bin"].max() + 1)
    M = np.full((nr, nc), np.nan)
    for _, row in grid.iterrows():
        M[int(row["r_bin"]), int(row["c_bin"])] = row["predicted_flow"]

    mask = np.isfinite(M)
    grand = np.nanmean(M)
    a = np.nanmean(M - grand, axis=1)
    b = np.nanmean(M - grand, axis=0)

    M_hat = grand + a[:, None] + b[None, :]
    ss_tot = np.nansum((M - np.nanmean(M))**2)
    ss_res = np.nansum((M - M_hat)**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return M, M_hat, r2

M, M_add, add_r2 = fit_additive_model(focus_df)
print("Additive separability grid R^2:", round(add_r2, 4))


In [ ]:

# ----------------------------
# Low-rank matrix approximation
# ----------------------------

def low_rank_explained_variance(M):
    X = M.copy()
    row_means = np.nanmean(X, axis=1, keepdims=True)
    inds = np.where(~np.isfinite(X))
    X[inds] = np.take(row_means.flatten(), inds[0])
    X = X - X.mean()
    U, S, VT = np.linalg.svd(X, full_matrices=False)
    var = S**2
    frac = var / var.sum() if var.sum() > 0 else var
    return S, frac, U, VT

S, frac, U, VT = low_rank_explained_variance(M)
display(pd.DataFrame({"singular_value": S[:8], "variance_fraction": frac[:8]}))


In [ ]:

fig = plt.figure(figsize=(14,4))

plt.subplot(1,3,1)
plt.imshow(M, aspect="auto", origin="lower")
plt.title("Observed gridded flow")
plt.colorbar()

plt.subplot(1,3,2)
plt.imshow(M_add, aspect="auto", origin="lower")
plt.title(f"Additive approximation\nR²={add_r2:.3f}")
plt.colorbar()

plt.subplot(1,3,3)
plt.plot(np.arange(1, min(11, len(frac)+1)), frac[:10], marker="o")
plt.xlabel("rank component")
plt.ylabel("variance fraction")
plt.title("Low-rank flow structure")
plt.tight_layout()
plt.show()



## Integration-based evaluation

A governing equation is more useful if it reproduces **trajectory structure**, not merely pointwise flow values.

We evaluate each surrogate by integrating:

\[
r_{t+1} = r_t + \Delta c \, g(r_t, c_t)
\]

along a canonical condition grid and comparing forward/backward terminal states.


In [ ]:

# ----------------------------
# Canonical grid for integration
# ----------------------------

c_min = float(focus_df["condition_coord"].quantile(0.02))
c_max = float(focus_df["condition_coord"].quantile(0.98))
canonical_c = np.linspace(c_min, c_max, INTEGRATION_GRID_SIZE)
canonical_dc = np.diff(canonical_c)
r0_values = np.linspace(
    float(focus_df["residual"].quantile(0.05)),
    float(focus_df["residual"].quantile(0.95)),
    R0_COUNT
)

print("Condition grid:", (c_min, c_max), "points:", len(canonical_c))
print("Initial residual range:", (r0_values.min(), r0_values.max()), "count:", len(r0_values))


In [ ]:

def integrate_model(model, r0, c_grid, direction="forward"):
    if direction == "forward":
        cc = c_grid
    elif direction == "backward":
        cc = c_grid[::-1]
    else:
        raise ValueError("direction must be 'forward' or 'backward'")

    traj = [r0]
    r = float(r0)
    for i in range(len(cc)-1):
        c = cc[i]
        dc = cc[i+1] - cc[i]
        g = float(model.predict(np.array([[r, c]]))[0])
        r = r + dc * g
        traj.append(r)
    traj = np.array(traj)
    if direction == "backward":
        traj = traj[::-1]
    return traj

def empirical_terminal_map(df_slice, r0_values, c_grid):
    # crude empirical baseline by local averaging from synthetic/real slice
    tmp = df_slice.copy().sort_values(["condition_coord", "residual"])
    out = []
    for r0 in r0_values:
        r = float(r0)
        for i in range(len(c_grid)-1):
            c = c_grid[i]
            dc = c_grid[i+1] - c_grid[i]
            d2 = (tmp["condition_coord"] - c)**2 + (tmp["residual"] - r)**2
            idx = d2.nsmallest(18).index
            g = tmp.loc[idx, "predicted_flow"].mean()
            r = r + dc * g
        out.append(r)
    return np.array(out)

baseline_terminal = empirical_terminal_map(focus_df, r0_values, canonical_c)


In [ ]:

# ----------------------------
# Trajectory / terminal evaluation
# ----------------------------

integration_rows = []
trajectory_bank = {}

for name, result in model_results.items():
    model = result["model"]
    forward_terms = []
    backward_terms = []
    trajectories = []

    for r0 in r0_values:
        traj_f = integrate_model(model, r0, canonical_c, direction="forward")
        traj_b = integrate_model(model, r0, canonical_c, direction="backward")
        trajectories.append((r0, traj_f, traj_b))
        forward_terms.append(traj_f[-1])
        backward_terms.append(traj_b[0])

    forward_terms = np.array(forward_terms)
    backward_terms = np.array(backward_terms)
    trajectory_bank[name] = trajectories

    row = {
        "model_name": name,
        "forward_vs_baseline_r2": r2_score(baseline_terminal, forward_terms),
        "forward_vs_baseline_rmse": mean_squared_error(baseline_terminal, forward_terms) ** 0.5,
        "forward_backward_gap_mean": np.mean(np.abs(forward_terms - backward_terms)),
        "forward_terminal_span": float(forward_terms.max() - forward_terms.min()),
        "backward_terminal_span": float(backward_terms.max() - backward_terms.min()),
    }
    integration_rows.append(row)

integration_df = pd.DataFrame(integration_rows).sort_values(
    ["forward_vs_baseline_r2", "forward_backward_gap_mean"],
    ascending=[False, True]
)
display(integration_df)


In [ ]:

# Combined ranking
combo = results_df.merge(integration_df, on="model_name", how="left")
combo["score"] = (
    0.45 * combo["test_metrics_r2"] +
    0.35 * combo["forward_vs_baseline_r2"] -
    0.20 * combo["forward_backward_gap_mean"]
)
combo = combo.sort_values("score", ascending=False).reset_index(drop=True)
display(combo)


In [ ]:

best_model_name = combo.iloc[0]["model_name"]
best_model = model_results[best_model_name]["model"]
print("Best governing surrogate:", best_model_name)


In [ ]:

# Visualize terminal maps for top 4 models
top4 = list(combo["model_name"].head(4))

fig = plt.figure(figsize=(12, 8))
for i, name in enumerate(top4, 1):
    model = model_results[name]["model"]
    f_terms = []
    b_terms = []
    for r0 in r0_values:
        traj_f = integrate_model(model, r0, canonical_c, direction="forward")
        traj_b = integrate_model(model, r0, canonical_c, direction="backward")
        f_terms.append(traj_f[-1])
        b_terms.append(traj_b[0])

    plt.subplot(2,2,i)
    plt.plot(r0_values, baseline_terminal, label="empirical baseline", linewidth=2)
    plt.plot(r0_values, f_terms, label="forward terminal")
    plt.plot(r0_values, b_terms, label="backward terminal")
    plt.xlabel("initial residual r0")
    plt.ylabel("terminal residual")
    plt.title(name)
    plt.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:

# Best-model trajectory family
best_trajs = trajectory_bank[best_model_name]

fig = plt.figure(figsize=(11,6))
for r0, traj_f, _ in best_trajs:
    plt.plot(canonical_c, traj_f, alpha=0.5)
plt.xlabel("condition coordinate c")
plt.ylabel("residual")
plt.title(f"Integrated trajectories — best surrogate: {best_model_name}")
plt.show()


In [ ]:

# Governing field portrait for best model
r_grid = np.linspace(float(focus_df["residual"].quantile(0.02)),
                     float(focus_df["residual"].quantile(0.98)), 55)
c_grid = np.linspace(c_min, c_max, 55)

CC, RR = np.meshgrid(c_grid, r_grid)
grid_X = np.column_stack([RR.ravel(), CC.ravel()])
GG = best_model.predict(grid_X).reshape(RR.shape)

fig = plt.figure(figsize=(10,6))
im = plt.imshow(GG, aspect="auto", origin="lower",
                extent=[c_min, c_max, r_grid.min(), r_grid.max()])
plt.colorbar(im, label="predicted g(r,c)")
plt.contour(CC, RR, GG, levels=[0], linewidths=2)
plt.xlabel("condition coordinate c")
plt.ylabel("residual r")
plt.title(f"Governing flow field — {best_model_name}")
plt.show()


In [ ]:

# Approximate symbolic readout for polynomial models when possible
def describe_model(name, fitted_model):
    if "poly" in name:
        poly = fitted_model.named_steps["poly"]
        reg = fitted_model.named_steps["reg"]
        feature_names = poly.get_feature_names_out(["r", "c"])
        coeffs = reg.coef_
        terms = sorted(zip(feature_names, coeffs), key=lambda x: abs(x[1]), reverse=True)
        return pd.DataFrame(terms[:12], columns=["term", "coefficient"])
    elif name == "linear":
        reg = fitted_model.named_steps["reg"]
        return pd.DataFrame({
            "term": ["r", "c"],
            "coefficient": reg.coef_
        })
    else:
        return pd.DataFrame({"term": ["non-symbolic surrogate"], "coefficient": [np.nan]})

display(describe_model(best_model_name, best_model))



## Interpretation guide

Use the summary table below to decide what Notebook 45 is saying.

- High **test R²** and high **integration R²**:
  the surrogate captures both local flow and global transport.

- High pointwise fit but weak integration:
  the model predicts local slopes but fails to preserve trajectory structure.

- Strong additive separability:
  a compact symbolic form \(a(r) + b(c)\) may be plausible.

- Strong low-rank structure:
  the field may be approximated by a small number of separable components.

- Large forward/backward terminal gap:
  the learned field preserves directional asymmetry and likely remains dissipative.


In [ ]:

summary = combo[[
    "model_name", "test_metrics_r2", "test_metrics_rmse", "test_metrics_mae",
    "forward_vs_baseline_r2", "forward_vs_baseline_rmse", "forward_backward_gap_mean", "score"
]].copy()

summary["additive_grid_r2"] = add_r2
summary["rank1_variance_fraction"] = frac[0] if len(frac) else np.nan
summary["rank2_cumulative_variance"] = frac[:2].sum() if len(frac) >= 2 else np.nan
display(summary)



## Suggested closing statement

A good closing line, once you inspect the winning model, is:

> Residual transport can be modeled by a compact surrogate governing field \(g(r,c)\) that preserves both local flow prediction and global trajectory structure, indicating that the mismatch is not random error but a learnable directional dynamical law.

If the best model is polynomial, the next notebook can target symbolic simplification.  
If the best model is kernel/MLP, the next notebook can target interpretable surrogate distillation.
